# Teste de Qualidade: OllamaRefiner (qwen2.5:3b vs 7b)

Compara qualidade de refinamento entre modelos antes de deployar no sidecar.

Criterios:
1. Preserva palavras originais (nao adiciona/remove)
2. Pontuacao correta
3. Capitalizacao adequada
4. Latencia aceitavel

In [ ]:
import asyncio
import time
import sys
sys.path.insert(0, '.')

from voice_ai.services.refiner import OllamaRefiner, RefineResult

SYSTEM_INSTRUCTION = (
    "Formate o texto transcrito: adicione pontuacao e capitalizacao. "
    "NAO mude nenhuma palavra, NAO resuma, NAO adicione conteudo. "
    "Retorne APENAS o texto formatado."
)

TEXTOS = {
    "informal_curto": (
        "entao doutor eu tava trabalhando normal ne ai fui demitido sem justa causa "
        "ne dai quero saber meus direitos"
    ),
    "juridico": (
        "o artigo 477 da clt determina que o empregador deve pagar as verbas rescisorias "
        "no prazo de dez dias contados a partir do termino do contrato inclusive o aviso "
        "previo indenizado o decimo terceiro proporcional e as ferias proporcionais "
        "acrescidas de um terco"
    ),
    "longo": (
        "bom dia doutor eu vim aqui porque eu trabalhei na empresa por cinco anos "
        "e ai eles me mandaram embora sem justa causa ne e na hora de pagar as contas "
        "eles nao pagaram direito eu acho que faltou o decimo terceiro e tambem as ferias "
        "que eu nao tirei e o fgts tambem nao sei se depositaram tudo certinho porque "
        "eu nunca conferi o extrato e agora eu to com medo de perder o prazo pra entrar "
        "com a acao porque ja faz uns seis meses que eu sai de la e o pessoal fala que "
        "tem um prazo de dois anos ne entao eu queria saber se ainda da tempo e quanto "
        "mais ou menos eu tenho pra receber porque eu preciso muito desse dinheiro"
    ),
}

print(f"Textos de teste: {list(TEXTOS.keys())}")
print(f"Ollama disponivel: {OllamaRefiner().is_available}")

In [ ]:
async def test_model(model: str) -> dict:
    """Testa um modelo com todos os textos."""
    refiner = OllamaRefiner()
    results = {}
    
    for nome, texto in TEXTOS.items():
        start = time.monotonic()
        result = await refiner.refine(
            text=texto,
            system_instruction=SYSTEM_INSTRUCTION,
            model=model,
            temperature=0.3,
        )
        elapsed = time.monotonic() - start
        
        # Conta palavras originais vs refinadas
        orig_words = set(texto.lower().split())
        ref_words = set(result.refined_text.lower().replace(',', '').replace('.', '').replace('?', '').split())
        added = ref_words - orig_words
        removed = orig_words - ref_words
        
        results[nome] = {
            "success": result.success,
            "latency_s": round(elapsed, 1),
            "added_words": added,
            "removed_words": removed,
            "refined": result.refined_text,
        }
        
        print(f"\n--- {nome} ({model}) [{elapsed:.1f}s] ---")
        print(f"Original:  {texto[:120]}...")
        print(f"Refinado:  {result.refined_text[:120]}...")
        if added:
            print(f"  ADICIONOU: {added}")
        if removed:
            print(f"  REMOVEU: {removed}")
    
    return results

print("Testando qwen2.5:3b...")
results_3b = await test_model("qwen2.5:3b")

In [ ]:
# Resumo comparativo
print("=" * 60)
print("RESUMO qwen2.5:3b")
print("=" * 60)
for nome, r in results_3b.items():
    status = "OK" if r["success"] and not r["added_words"] else "ATENCAO"
    print(f"  {nome}: {r['latency_s']}s | {status} | +{len(r['added_words'])} -{len(r['removed_words'])} palavras")

print("\nSe qualidade aceitavel, nao precisa testar 7b.")
print("Se nao, rode: ollama pull qwen2.5:7b e execute a celula abaixo.")

In [ ]:
# Descomente para testar 7b (precisa: ollama pull qwen2.5:7b)
# print("Testando qwen2.5:7b...")
# results_7b = await test_model("qwen2.5:7b")
# 
# print("\n" + "=" * 60)
# print("RESUMO qwen2.5:7b")
# print("=" * 60)
# for nome, r in results_7b.items():
#     status = "OK" if r["success"] and not r["added_words"] else "ATENCAO"
#     print(f"  {nome}: {r['latency_s']}s | {status} | +{len(r['added_words'])} -{len(r['removed_words'])} palavras")